In [ ]:
using Gridap
using GridapGmsh
using GridapEmbedded

In [2]:
# L for x axis
# B for y axis
# H for z axis

function Rectangular_Prism(;L=1, B=1, H=1, x0=Point(0,0,0), name="Rectangular_Prism")

  e1 = VectorValue(1,0,0)
  e2 = VectorValue(0,1,0)
  e3 = VectorValue(0,0,1)

  plane1 = plane(x0 = x0 - 0.5*H*e3, v=-e3, name="face1")
  plane2 = plane(x0 = x0 + 0.5*H*e3, v=+e3, name="face2")
  plane3 = plane(x0 = x0 - 0.5*B*e2, v=-e2, name="face3")
  plane4 = plane(x0 = x0 + 0.5*B*e2, v=+e2, name="face4")
  plane5 = plane(x0 = x0 - 0.5*L*e1, v=-e1, name="face5")
  plane6 = plane(x0 = x0 + 0.5*L*e1, v=+e1, name="face6")

  geo12 = intersect(plane1,plane2)
  geo34 = intersect(plane3,plane4)
  geo56 = intersect(plane5,plane6)

  intersect(intersect(geo12,geo34),geo56,name=name)

end

Rectangular_Prism (generic function with 1 method)

In [3]:
geo = Rectangular_Prism(
  L = 2.0,
  B = 1.0,
  H = 0.5,
  x0 = Point(0.0, 0.0, 0.0),
  name = "beam"
)

AnalyticalGeometry(Node((:∩, "beam", nothing),Node((:∩, "", nothing),Node((:∩, "", nothing),Leaf((GridapEmbedded.LevelSetCutters.var"#planefun#14"{VectorValue{3, Float64}, VectorValue{3, Int64}}((0.0, 0.0, -0.25), (0, 0, -1)), "face1", nothing)),Leaf((GridapEmbedded.LevelSetCutters.var"#planefun#14"{VectorValue{3, Float64}, VectorValue{3, Int64}}((0.0, 0.0, 0.25), (0, 0, 1)), "face2", nothing))),Node((:∩, "", nothing),Leaf((GridapEmbedded.LevelSetCutters.var"#planefun#14"{VectorValue{3, Float64}, VectorValue{3, Int64}}((0.0, -0.5, 0.0), (0, -1, 0)), "face3", nothing)),Leaf((GridapEmbedded.LevelSetCutters.var"#planefun#14"{VectorValue{3, Float64}, VectorValue{3, Int64}}((0.0, 0.5, 0.0), (0, 1, 0)), "face4", nothing)))),Node((:∩, "", nothing),Leaf((GridapEmbedded.LevelSetCutters.var"#planefun#14"{VectorValue{3, Float64}, VectorValue{3, Int64}}((-1.0, 0.0, 0.0), (-1, 0, 0)), "face5", nothing)),Leaf((GridapEmbedded.LevelSetCutters.var"#planefun#14"{VectorValue{3, Float64}, VectorValue{3, I

In [7]:
# Domain = (x_min, x_max, y_min, y_max, z_min, z_max)
domain = (-2, 2, -2, 2, -2, 2)
partition = (50, 50, 50)

bgmodel = CartesianDiscreteModel(domain, partition)

CartesianDiscreteModel()

In [8]:
# Background Model , Analytical Geometry
cutgeo = cut(bgmodel,geo)

EmbeddedDiscretization()

In [10]:
Ω_act = Triangulation(cutgeo,ACTIVE)
Ω_bg = Triangulation(bgmodel)

BodyFittedTriangulation()

In [11]:
result_path = joinpath(@__DIR__, "..", "..", "Result", "Unfitted_FEM","3D_Plate_With_Hole_CutFEM")
isdir(result_path) || mkpath(result_path)

writevtk(Ω_act, joinpath(result_path,"ACTIVE_Triangulation"))
writevtk(Ω_bg, joinpath(result_path,"Background_Triangulation"))

(["C:\\Users\\IIT BBSR\\Desktop\\Amiya\\BTP\\Unfitted_FEM_Prolems\\..\\..\\Result\\Unfitted_FEM\\3D_Plate_With_Hole_CutFEM\\Background_Triangulation.vtu"],)

In [12]:
Ω = Triangulation(cutgeo,PHYSICAL)
writevtk(Ω, joinpath(result_path,"Physical_Triangulation"))

(["C:\\Users\\IIT BBSR\\Desktop\\Amiya\\BTP\\Unfitted_FEM_Prolems\\..\\..\\Result\\Unfitted_FEM\\3D_Plate_With_Hole_CutFEM\\Physical_Triangulation.vtu"],)

In [13]:
order = 1
reffe = ReferenceFE(lagrangian,Float64,order)
Vstd = TestFESpace(Ω_act,reffe,conformity=:H1)

UnconstrainedFESpace()

In [14]:
strategy = AggregateAllCutCells()
aggregates = aggregate(strategy,cutgeo);

In [16]:
colors = color_aggregates(aggregates,bgmodel)
Ω_bg = Triangulation(bgmodel)

writevtk(
    Ω_bg,
    joinpath(result_path, "Aggs_on_BG_Triangulation");
    celldata = ["aggregate" => aggregates, "color" => colors]
)

(["C:\\Users\\IIT BBSR\\Desktop\\Amiya\\BTP\\Unfitted_FEM_Prolems\\..\\..\\Result\\Unfitted_FEM\\3D_Plate_With_Hole_CutFEM\\Aggs_on_BG_Triangulation.vtu"],)

In [17]:
V = AgFEMSpace(Vstd,aggregates)
U = TrialFESpace(V)

FESpaceWithLinearConstraints()

In [18]:
degree = 2*order
dΩ = Measure(Ω,degree)

GenericMeasure()

In [19]:
Γ = EmbeddedBoundary(cutgeo)
n_Γ = get_normal_vector(Γ)
dΓ = Measure(Γ,degree)

GenericMeasure()

In [22]:
u(x) = x[1] - x[2] # Solution of the problem
const γd = 10.0    # Nitsche coefficient
const h = dp[1]/n  # Mesh size according to the parameters of the background grid

a(u,v) =
  ∫( ∇(v)⋅∇(u) )dΩ +
  ∫( (γd/h)*v*u  - v*(n_Γ⋅∇(u)) - (n_Γ⋅∇(v))*u )dΓ

l(v) = ∫( (γd/h)*v*u - (n_Γ⋅∇(v))*u )dΓ

LoadError: UndefVarError: `dp` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [21]:
g = VectorValue(100.0, 0.0, 0.0)                # Neumann Boundary Condition
σ(ε) = λ*tr(ε)*one(ε) + 2*μ*ε                   # Sress Tensor

σ (generic function with 1 method)